# **Image to Tabular Data Pipeline**


*   List images from files
*   Do feature detections and feature extraction
*   Input those features into the table
*   Make data table into CVS and share

### **Idea is to make this pipeline understandable and readable to everyone to be able to use to be able to make features using OpenCV for image data**





---



## Retriving Images from Google Drive



1.   Using a loop (GLOB) I will retieve the images and path ways in the Google Drive Folder.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Folder with Training Images
Folder_Path_Test = "/content/drive/MyDrive/CAPSTONE PROJECT/Test_Full"
Folder_Path_Train = "/content/drive/MyDrive/CAPSTONE PROJECT/Train_Full"

### Find image paths for full and test and combined those images and get the labels for those images

In [ ]:
#Libraries
import os
import glob
from PIL import Image
import pandas as pd
import numpy as np
import cv2
from skimage.color import rgb2gray
!pip install mahotas
import mahotas as mh

In [ ]:
from pathlib import Path

def get_all_image_paths(directory_path):
    """
    Recursively finds all image file paths in the specified directory and its subdirectories.

    Args:
        directory_path (str or Path): The path to the top-level directory.

    Returns:
        list: A list of Path objects for all found images.
    """
    # Define common image extensions
    image_extensions = ('*.png', '*.jpg', '*.jpeg', '*.gif', '*.bmp')
    all_image_paths_local = []

    # Use rglob (recursive glob) to find files matching the extensions
    for ext in image_extensions:
        all_image_paths_local.extend(Path(directory_path).rglob(ext))

    return all_image_paths_local

# Re-create images_list_test to ensure it's defined
main_folder_test = Folder_Path_Test
images_list_test = get_all_image_paths(main_folder_test)
print(f"Found {len(images_list_test)} images in test folder.")

# Re-create images_list_train to ensure it's defined
second_folder_train = Folder_Path_Train
images_list_train = get_all_image_paths(second_folder_train)
print(f"Found {len(images_list_train)} images in train folder.")

# Combine the lists as originally intended
all_image_paths = images_list_test + images_list_train
print(f"Total images found: {len(all_image_paths)}")

Found 118 images in test folder.
Found 2239 images in train folder.
Total images found: 2357


In [ ]:
image_labels = []

for image_path in all_image_paths:
    label = image_path.parent.name
    image_labels.append(label)

print(f"Total labels collected: {len(image_labels)}")

Total labels collected: 2357



To extract the class labels, I will iterate through the all_image_paths list, and for each path, use the parent.name attribute to get the directory name, which serves as the class label. These labels will be stored in a new list called image_labels.



# **Extracting Features Using OpenCV**

In [ ]:
import math

num_bins = 32

def extract_features(image_path):
    img_pil = Image.open(image_path).convert("RGB")
    img_array = np.array(img_pil)

    # Extracting simple pixel-based features: average RGB values
    avg_r = np.mean(img_array[:, :, 0])
    avg_g = np.mean(img_array[:, :, 1])
    avg_b = np.mean(img_array[:, :, 2])

    # Extracting color histogram features using OpenCV
    # Convert PIL image to OpenCV format (NumPy array in BGR order)
    img_cv2 = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)

    # Calculate histograms for each color channel with fewer bins
    hist_b = cv2.calcHist([img_cv2], [0], None, [num_bins], [0, 256])
    hist_g = cv2.calcHist([img_cv2], [1], None, [num_bins], [0, 256])
    hist_r = cv2.calcHist([img_cv2], [2], None, [num_bins], [0, 256])

    # Flatten the histograms and concatenate them
    histogram_features = np.concatenate((hist_b, hist_g, hist_r)).flatten()

    # Extracting Haralick texture features
    # Convert to grayscale for Haralick features
    gray_img = rgb2gray(img_array)
    gray_img_uint8 = (gray_img * 255).astype(np.uint8)

    # Calculate Haralick features
    haralick_features = mh.features.haralick(gray_img_uint8).mean(0) # Calculate mean across distances/angles

    # Initialize lists for RGB channel-wise features
    canny_features_rgb = []
    gaussian_features_rgb = []
    laplacian_features_rgb = []
    sobel_features_rgb = []

    for i in range(3): # Iterate over R, G, B channels
        channel = img_array[:, :, i] # Get the current channel
        channel_uint8 = channel.astype(np.uint8)

        # Canny Edge Detector features for each channel
        canny_edges = cv2.Canny(channel_uint8, 100, 200)
        canny_features_rgb.append(np.mean(canny_edges))

        # Gaussian Blur features for each channel
        gaussian_blur = cv2.GaussianBlur(channel_uint8, (5, 5), 0)
        gaussian_features_rgb.append(np.mean(gaussian_blur))

        # Laplacian features for each channel
        laplacian = cv2.Laplacian(channel_uint8, cv2.CV_64F)
        laplacian_features_rgb.append(np.mean(np.abs(laplacian)))

        # Sobel Edge detection features for each channel
        sobelx = cv2.Sobel(channel_uint8, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(channel_uint8, cv2.CV_64F, 0, 1, ksize=3)
        sobel_magnitude = np.sqrt(sobelx**2 + sobely**2)
        sobel_features_rgb.append(np.mean(sobel_magnitude))

    # --- New Shape Features: minEnclosingCircle and minAreaRect ---
    # Find contours from the grayscale image (Canny edges are good for this)
    # Using RETR_EXTERNAL to get only external contours, CHAIN_APPROX_SIMPLE to save memory
    contours, _ = cv2.findContours(gray_img_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Initialize shape features with default values (0) if no contours are found
    min_enc_circle_radius = 0.0
    min_enc_circle_area = 0.0
    min_area_rect_width = 0.0
    min_area_rect_height = 0.0
    min_area_rect_aspect_ratio = 0.0
    min_area_rect_angle = 0.0

    if contours:
        # Find the largest contour by area
        largest_contour = max(contours, key=cv2.contourArea)

        # Features from cv2.minEnclosingCircle
        (x_circle, y_circle), radius = cv2.minEnclosingCircle(largest_contour)
        min_enc_circle_radius = radius
        min_enc_circle_area = math.pi * (radius**2)

        # Features from cv2.minAreaRect
        rect = cv2.minAreaRect(largest_contour)
        # rect is (center(x,y), size(width, height), angle)
        min_area_rect_width = rect[1][0]
        min_area_rect_height = rect[1][1]
        min_area_rect_angle = rect[2]

        # Calculate aspect ratio, avoiding division by zero
        if min_area_rect_height > 0:
            min_area_rect_aspect_ratio = min_area_rect_width / min_area_rect_height
        elif min_area_rect_width > 0:
            min_area_rect_aspect_ratio = min_area_rect_height / min_area_rect_width # handle case where width is greater


    # Combine all features
    all_features = [avg_r, avg_g, avg_b] + \
                   histogram_features.tolist() + \
                   haralick_features.tolist() + \
                   canny_features_rgb + \
                   gaussian_features_rgb + \
                   laplacian_features_rgb + \
                   sobel_features_rgb + \
                   [min_enc_circle_radius, min_enc_circle_area, \
                    min_area_rect_width, min_area_rect_height, \
                    min_area_rect_aspect_ratio, min_area_rect_angle]

    return all_features

print("extract_features function updated successfully for RGB channels and new shape features.")

extract_features function updated successfully for RGB channels and new shape features.


Apply these operations to each channel (Red, Green, Blue) separately because many image processing techniques, especially those for edge detection and texture analysis like Canny, Laplacian, and Sobel, are fundamentally designed to work on single-channel (grayscale) intensity images. When you work with a color image, you essentially have three intensity channels (one for each primary color). By applying these operations to each channel independently, you allow the algorithm to detect features specific to the variations within that color component. For example, a strong red edge might not be as apparent in the green or blue channels, or in a combined grayscale version where the red contrast is diluted.

Processing each channel individually allows for:
 * Richness of Information: You capture features that are distinct to each color component.
 * Preservation of Color-Specific Features: Edges or textures that are primarily defined by changes in one color channel are better preserved and detected.
 * Consistency: It's a systematic way to extend single-channel algorithms to multi-channel images, treating each color plane as an independent intensity map.

 If you were to convert the image to grayscale first, you'd lose some of this color-specific information, as the grayscale conversion averages the color channels into a single intensity value. By processing each RGB channel, you get a more comprehensive set of features reflecting color-dependent characteristics.

In [ ]:
features_list = []

for path in all_image_paths:
    features_list.append(extract_features(path))

# Determine the column names.
num_hist_features = num_bins * 3
num_haralick_features = 13 # Mahotas Haralick returns 13 features by default
num_new_features_per_method = 3 # R, G, B channels for Canny, Gaussian, Laplacian, Sobel
num_shape_features = 6 # radius, area, width, height, aspect_ratio, angle

feature_column_names = ["avg_r", "avg_g", "avg_b"] + \
                      [f"hist_feature_{i}" for i in range(num_hist_features)] + \
                      [f"haralick_feature_{i}" for i in range(num_haralick_features)] + \
                      [f"canny_r", f"canny_g", f"canny_b"] + \
                      [f"gaussian_r", f"gaussian_g", f"gaussian_b"] + \
                      [f"laplacian_r", f"laplacian_g", f"laplacian_b"] + \
                      [f"sobel_r", f"sobel_g", f"sobel_b"] + \
                      ["min_enc_circle_radius", "min_enc_circle_area", \
                       "min_area_rect_width", "min_area_rect_height", \
                       "min_area_rect_aspect_ratio", "min_area_rect_angle"]

df = pd.DataFrame(features_list, columns=feature_column_names)
df["label"] = image_labels # Add the image_labels list as a new column
print(df.head())
print(df.shape)

        avg_r       avg_g       avg_b  hist_feature_0  hist_feature_1  \
0  215.435722  154.768741  160.486663             0.0             0.0   
1  203.781219  179.247785  208.150415             0.0             0.0   
2  195.706115  120.804822  142.253215             0.0             0.0   
3  144.887641  136.410988  136.469449         32976.0         19025.0   
4  134.110788  147.433253  167.797929          5904.0          5218.0   

   hist_feature_2  hist_feature_3  hist_feature_4  hist_feature_5  \
0             0.0             0.0             1.0             1.0   
1             0.0             0.0             1.0             2.0   
2             0.0             0.0             0.0             0.0   
3          7816.0         15109.0         10365.0          7888.0   
4          1957.0          1837.0          2940.0          5767.0   

   hist_feature_6  ...    sobel_r    sobel_g    sobel_b  \
0             0.0  ...  16.298068  15.719409  23.341190   
1             4.0  ...  14.1

### Let's break down the features extracted:

**avg_r, avg_g, avg_b (Average RGB Values):**
*  These are the most basic color features. They represent the average intensity of the Red, Green, and Blue channels across the entire image. They give a general sense of the image's overall color tone.

**hist_feature_0 to hist_feature_95 (Color Histogram Features):**
* These features represent the color distribution within the image. For each of the Red, Green, and Blue channels, a histogram is calculated with num_bins (which is 32) bins. Each hist_feature_X represents the count of pixels falling into a specific intensity range for a particular color channel. For example, hist_feature_0 to hist_feature_31 could be for Red, hist_feature_32 to hist_feature_63 for Green, and hist_feature_64 to hist_feature_95 for Blue. They provide a more detailed description of the color composition than just the average values.

**haralick_feature_0 to haralick_feature_12 (Haralick Texture Features):**
* These are a set of 13 statistical measures derived from the Gray-Level Co-occurrence Matrix (GLCM) of the grayscale version of the image. They quantify various aspects of image texture, such as contrast, correlation, energy, homogeneity, etc. They are very useful for distinguishing regions based on their visual texture.

**canny_r, canny_g, canny_b (Canny Edge Detector Features):**
* These features represent the mean intensity of edges detected by the Canny algorithm in the Red, Green, and Blue channels, respectively. Canny is a multi-stage algorithm designed to detect a wide range of edges. A higher value suggests more pronounced edges in that specific color channel.

**gaussian_r, gaussian_g, gaussian_b (Gaussian Blur Features):**
* These represent the mean pixel intensity of the Red, Green, and Blue channels after applying a Gaussian blur. Gaussian blur is often used to reduce image noise and detail. The mean value after blurring can give an indication of the overall smoothness or dominant intensity in each channel.

**laplacian_r, laplacian_g, laplacian_b (Laplacian Features):**
* These features represent the mean absolute value of the Laplacian operator applied to the Red, Green, and Blue channels. The Laplacian is an edge detector that highlights regions of rapid intensity change. A higher value indicates more frequent or stronger intensity changes within that color channel.

**sobel_r, sobel_g, sobel_b (Sobel Edge Detection Features):**
* These features represent the mean magnitude of the Sobel edge detection operator applied to the Red, Green, and Blue channels. The Sobel operator calculates the gradient magnitude and direction. The mean magnitude for each channel indicates the average strength of edges detected in that color component.

**min_enc_circle_radius (Minimum Enclosing Circle Radius):**
* This is the radius of the smallest circle that completely encloses the largest detected contour in the image. It's a measure of the overall size of the main object in the image, assuming the largest contour represents the primary object.

**min_enc_circle_area (Minimum Enclosing Circle Area):**
 * This is the area of the minimum enclosing circle. Similar to the radius, it provides a size metric for the largest contour.

**min_area_rect_width, min_area_rect_height (Minimum Area Rectangle Dimensions):**
* These are the width and height of the smallest oriented bounding rectangle that encloses the largest detected contour. They provide information about the dimensions of the main object, regardless of its orientation.

**min_area_rect_aspect_ratio (Minimum Area Rectangle Aspect Ratio):**
* This is the ratio of the width to the height (or height to width, depending on implementation) of the minimum area rectangle. It describes the elongatedness of the main object.

**min_area_rect_angle (Minimum Area Rectangle Angle):**
* This is the angle of the minimum area rectangle with respect to the x-axis. It gives information about the orientation of the main object in the image.

**label (Image Label):**
* This is the class label associated with each image, extracted from its folder name. This column is your target variable for classification tasks.

In [ ]:
from google.colab import files

output_csv_filename = 'SC_Dataset_9_Classes.csv'
df.to_csv(output_csv_filename, index=False)

print(f"DataFrame successfully saved to '{output_csv_filename}'")

# Provide a download link
files.download(output_csv_filename)

DataFrame successfully saved to 'SC_Dataset_9_Classes.csv'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>